# 6. Mixed-integer linear programme

Lecture: *Week 3 - Linear Programming 2*, PDF pages 24–25 (*Mixed-Integer Program* and
*Mixed-Integer Solution (problem based)*).

**The problem.** The same problem as demo 5, but only $x$ must be a whole number; $y$ may be any value $\ge 0$.
In the code, $x$ is `x1` and $y$ is `x2`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** $x = 2$, $y = 2.833$, $P = 13.917$: between the integer answer (13) and the LP answer
(13.95), because the problem is less restricted than demo 5 and more restricted than the LP.

*Original script: `5 Source Files/Python/milp_pyomo.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

One integer and one continuous variable: that is what makes it *mixed*-integer. The `bounds=(0, inf)` repeat what the *NonNegative* domains already say.

In [ ]:
# Decision variables: x1 (= x) whole number, x2 (= y) any real value
model.x1 = Var(domain=NonNegativeIntegers, bounds=(0, float('inf')))
model.x2 = Var(domain=NonNegativeReals, bounds=(0, float('inf')))

The objective $P = 2x + 3.5y$, maximised.

In [ ]:
# Objective function (maximize 2x1 + 3.5x2)
f = [2, 3.5]
model.obj = Objective(expr=f[0]*model.x1 + f[1]*model.x2, sense=maximize)

The three constraints, as in demo 5.

In [ ]:
# Constraints
model.c1 = Constraint(expr=3*model.x1 + 2*model.x2 <= 14)    # 3x + 2y <= 14
model.c2 = Constraint(expr=2*model.x1 + 3*model.x2 <= 12.5)  # 2x + 3y <= 12.5
model.c3 = Constraint(expr=-model.x1 + model.x2 <= 1)        # -x + y <= 1

Print the model to check it before solving.

In [ ]:
# Print model before solving
print("Model formulation:")
model.pprint()

Solve with HiGHS (it handles integer variables by branch and bound).

In [ ]:
# Solve with HiGHS
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)

If the solver reports *optimal*, display the variables, the objective and the constraints.

In [ ]:
# Check results
if (results.solver.status == SolverStatus.ok) and \
   (results.solver.termination_condition == TerminationCondition.optimal):
    print("\nOptimal solution found:")
    model.display()
else:
    print("\nSolver did not find an optimal solution. Status:", results.solver.status)

## What do you notice?

Which constraint stops $y$ from growing once $x = 2$ is fixed?

<details><summary>Answer</summary>

`c2`: its *Body* is 12.5, equal to its bound, so $2 \cdot 2 + 3y = 12.5$ gives $y = 8.5/3 = 2.833$.
`c1` (11.67 < 14) and `c3` (0.83 < 1) have spare.
</details>